![Databricks Academy](./Includes/images/common/db-academy.png)

# 3 - Adicionando Expectativas de Qualidade de Dados

Nesta demonstração, vamos adicionar expectativas de qualidade de dados para aplicar restrições de qualidade que validam os dados à medida que fluem pelos Pipelines Declarativos Spark Lakeflow. As expectativas fornecem maior insight sobre métricas de qualidade de dados e permitem que você falhe atualizações ou descarte registros ao detectar registros inválidos.

### Objetivos de Aprendizagem

Ao final desta lição, você será capaz de:
- Adicionar restrições de qualidade dentro de um Pipeline Declarativo Spark Lakeflow para acionar ações apropriadas (alerta, descarte ou falha) com base nas expectativas de dados.
- Analisar métricas do pipeline para identificar e interpretar problemas de qualidade de dados em diferentes fluxos de dados.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the following cell to configure your working environment for this course.

    This cell will also reset your `/Volumes/labuser/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 10.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-f14cbf79-9445-4a47-aa23-2d8667cc5c69
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-f14cbf79-9445-4a47-aa23-2d8667cc5c69
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser15140516_1778971530'.



  STEP 1: Verifying catalog exists: labuser15140516_1778971530
  Catalog 'labuser15140516_1778971530' exists.

  STEP 2: Setting up 3 schema(s) in catalog: labuser15140516_1778971530
  [1/3] Checking: `labuser15140516_1778971530`.`sdp_1_bronze`... ALREADY EXISTS
  [2/3] Checking: `labuser15140516_1778971530`.`sdp_2_silver`... ALREADY EXISTS
  [3/3] Checking: `labuser15140516_1778971530`.`sdp_3_gold`... ALREADY EXISTS

  COMPLETE: 0 schema(s) created, 3 already existed.



DataFrame[]


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating volume folder path...
  Found: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers

  STEP 2: Scanning for files...
  Found 1 file(s) to delete.

  STEP 3: Deleting files from: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers
  [1/1] Deleting: 00.json... DELETED

  COMPLETE: Deleted 1 file(s) from /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/la

Information,Value
Your Catalog:,labuser15140516_1778971530
Bronze Schema:,sdp_1_bronze
Silver Schema:,sdp_2_silver
Gold Schema:,sdp_3_gold
Source Volume:,/Volumes/labuser15140516_1778971530/sdp_1_bronze/source


Compute,Status,Details
Serverless,✓ Match,Version 5


2. Run the cell below to programmatically view the files in your `/Volumes/labuser_USERNAME/sdp_1_bronze/source/orders/` volume.

    Confirm you only see the original **00.json** file in the **orders** folder.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/orders"').display()

path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders/00.json,00.json,15313,1779030991000


## B. Adding Data Quality Expectations

This demonstration includes the simple starter Spark Declarative Pipeline that has already been created in the previous demonstration.

  We will continue to build on it to explore its capabilities.

  **Manage data quality with pipeline expectations**:
[AWS](https://docs.databricks.com/aws/en/ldp/expectations) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/expectations) |
[GCP](https://docs.databricks.com/gcp/en/ldp/expectations)


1. Run the cell below to create your starter pipeline for this demonstration (pipeline from the previous demonstration).

    The setup code will set the following for you:

    - Your default catalog: **labuser**

    - Your configuration parameter: `source` = `/Volumes/labuser_USERNAME/sdp_1_bronze/source/`

      **NOTE:** If the pipeline already exists, an error will be returned. In that case, you'll need to delete the existing pipeline and rerun this cell.

      To delete the pipeline:

      - Select **Jobs and Pipelines** from the far-left navigation bar.

      - Find the pipeline you want to delete.

      - Click the three-dot menu ![ellipsis icon](./Includes/images/common/ellipsis_icon.png).

      - Select **Delete**.

**NOTE:**  The `create_declarative_pipeline` function is a custom function built for this course to create the sample pipeline using the Databricks REST API. This avoids manually creating the pipeline and referencing the pipeline assets.

In [0]:
%python
create_declarative_pipeline(
    pipeline_name=f'3 - Adding Data Quality Expectations Project - {my_catalog}',
    root_path_folder_name='3 - Adding Data Quality Expectations Project',
    catalog_name=my_catalog,
    schema_name='default',
    source_folder_names=['orders'],
    configuration={'source': source_volume_path}
)


  STEP 1: Checking for existing pipeline...
  ✅ No existing pipeline named '3 - Adding Data Quality Expectations Project - labuser15140516_1778971530' found.

  STEP 2: Building pipeline configuration...
  Pipeline Name:    3 - Adding Data Quality Expectations Project - labuser15140516_1778971530
  Catalog:          labuser15140516_1778971530
  Schema:           default
  Root Path:        /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/3 - Adding Data Quality Expectations Project
  Source Folders:    orders
  Serverless:       True
  Photon:           True
  Channel:          CURRENT
  Continuous:       False
  Development Mode: True
  Configuration:
    source = /Volumes/labuser15140516_1778971530/sdp_1_bronze/source

  STEP 3: Creating pipeline via API...
  ✅ Pipeline '3 - Adding Data Quality Expectations Project - labuser15140516_177897

2. Complete os seguintes passos para abrir o projeto inicial do Spark Declarative Pipeline para esta demonstração:

   a. Na barra de navegação principal, clique com o botão direito em **Jobs & Pipelines** e selecione **Abrir link em nova guia**.

   b. Em **Jobs & Pipelines**, selecione seu pipeline **3 - Adding Data Quality Expectations Project - labuser**.
      - **REQUERIDO:** No topo, próximo ao nome do pipeline, ative o **Novo monitoramento de pipeline**.

   c. No painel **Detalhes do Pipeline** à direita, selecione **Abrir no Editor** (campo à direita de **Código-fonte**) para abrir o pipeline no **Lakeflow Pipeline Editor**.

   d. Na nova guia:
      - Selecione a pasta **orders** (A pasta principal também contém a pasta extra **python_excluded** que contém a versão em Python)

      - Clique em **orders_pipeline.sql**.

## C. Explore and Run the `orders_pipeline.sql` Pipeline with Data Quality Expectations

1. Select the **Run pipeline** button to run the pipeline.

    *While the pipeline is running, proceed to step 2.*

2. While the pipeline is executing:

   a. Examine `CREATE OR REFRESH STREAMING TABLE 2_silver_db.orders_silver_demo3` (Section **Bronze -> Silver (Contains Data Quality Expectations)**)

   b. Notice that it includes **3 data quality expectations** applied as data is ingested into **orders_silver_demo3**:

      | Constraint | Rule | Action |
      |------------|------|--------|
      | `valid_notifications` | `notifications` must be `'Y'` or `'x'` | **Warn** — rows are kept, violation is logged |
      | `valid_date` | `order_timestamp` must be after `'2021-12-26'` | **Drop Row** — invalid rows are removed |
      | `valid_id` | `customer_id` must not be `NULL` | **Fail Update** — pipeline fails if violated |

      <br></br>
      - **notifications** column only contains `Y` or `N` values - so `N` values will trigger a warning
      - **order_timestamp** column contains dates on `2021-12-25` - so those rows will be dropped
      - **customer_id** has no nulls in this demo - so the pipeline will pass


## D. Explore a Execução do Pipeline

1. Após a conclusão do pipeline, explore o **Gráfico do Pipeline** à direita.

   - Ele cria o pipeline:
      - **orders_bronze_demo3** > **orders_silver_demo3** > **gold_orders_by_date_demo3**

   - Observe:
      - **174 linhas** foram lidas na tabela bronze
      - Apenas **148 linhas** foram lidas na tabela silver (a tabela com restrições)

2. Na janela inferior, certifique-se de estar na guia **Tabelas**.

   - Selecione **orders_silver_demo3** e depois selecione **Métricas da tabela**

   - Observe o seguinte na tabela:

 Métrica | Valor | Descrição |
---------|-------|-----------|
 **Registros de saída** | 148 | Linhas que passaram todas as expectativas e foram gravadas na tabela |
 **Expectativas** | 1 atendida \| 2 não atendidas | Total de expectativas de qualidade de dados definidas na tabela streaming |
 **Descartados** | 26 | Linhas que falharam na expectativa `DROP ROW` (taxa de falha de 14,9%) |
 **Avisos** | 32 | Linhas que falharam na expectativa `WARN` (taxa de falha de 14,9%) |

   - Selecione o link na coluna **Expectativas** para ver a análise detalhada:

 Restrição | Ação | Taxa de Falha | Linhas Falhadas |
-----------|------|---------------|-----------------|
 `valid_notifications` | **Permitir** (aviso) | 22,4% | 39 |
 `valid_date` | **Descartar** | 14,9% | 26 |


- **NOTAS:**
  - Se as contagens de `WARN` diferirem entre a tabela e o popup, é porque a visualização da tabela desduplicada linhas sobrepostas, enquanto o popup conta falhas por expectativa, mesmo quando a mesma linha falha em várias expectativas.
  - As métricas de expectativa na interface são por execução de atualização. Para analisar a qualidade dos dados em várias execuções, consulte o log de eventos do pipeline (abordado mais adiante no curso).

#### Checkpoint
![](./Includes/images/data-quality-expectations/quality-expectations-run.png)

## Additional Resources

- Manage data quality with pipeline expectations:
[AWS](https://docs.databricks.com/aws/en/dlt/expectations) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/expectations) |
[GCP](https://docs.databricks.com/gcp/en/dlt/expectations)

- Expectation recommendations and advanced patterns:
[AWS](https://docs.databricks.com/aws/en/dlt/expectation-patterns) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/expectation-patterns) |
[GCP](https://docs.databricks.com/gcp/en/dlt/expectation-patterns)

- [Data Quality Management With Databricks](https://www.databricks.com/discover/pages/data-quality-management)


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>